# 03 · Metabolic metacells: build and benchmark (Kolla et al. 2020, E16)

**Part 1** builds metacells with `metabolic_metacells`:
- metacells never mix cell types **or samples**;
- size is set by a **UMI budget** per cell type, the pool needed to detect 80% of reachable reaction-relevant genes (negative binomial dropout model), capped so each cell type keeps at least 3 metacells;
- a cell type × sample with fewer UMIs than the budget becomes **one metacell flagged `under_budget`**. A split that would leave a metacell short is retried with one group fewer, so the flag means the stratum was too small, not that the split was uneven;
- genes and reactions are classed per cell type as **pooled / partial / uncertain / off**. `partial` means detectable only in a metacell larger than the one built — seen in some metacells and not others — so it is neither trustworthy nor hopeless, and how far to trust it is a decision for the flux step.

It builds **four sizing variants**, crossing `split_isozymes` (size for each isozyme branch, or for the summed reaction) with `sizing_and_strategy` (`min` sizes for every complex subunit, `median` for the typical one). Which is right depends on how detectable this tissue's complex subunits and minor isozymes are, so it is measured rather than assumed.

**Part 2** benchmarks them by thinning UMIs (simulating shallower sequencing) and asking how well each grouping recovers its own full-depth reaction scores. Every variant is **scored with the same settings** (`SCORING_SPLIT_ISOZYMES`), or the columns would not be comparable. References:
- `fixed_size_matched`: same number of metacells per cell type as the reference variant, fixed cells per metacell, samples mixed. Isolates the effect of UMI budgeting and sample separation.
- `fixed_50`: 50 cells per metacell, samples mixed (the current SEACells setting).
- `seacells_50`: real SEACells, if installed and `RUN_SEACELLS = True`.

**Part 3** looks specifically at **under-budget metacells** in the reference variant: how much sparser are they, and does the dropout model predict it?

## 0 · Settings

In [ ]:
import os, sys, json

BASE = '/scratch/prj/crb_inner_ear/k2147692/metabolic'
REPO_DIR = f'{BASE}/code/Metabolic-pipeline'
DATA_PATH = f'{BASE}/data/kolla/kolla_E16.h5ad'
OUT_DIR = f'{BASE}/results/03_metabolic_metacells_E16'

CELLTYPE_COL = 'cell_type'
SAMPLE_COL = 'sample'
SYMBOL_COL = 'gene_symbol'
SPECIES = 'mmusculus'
AND_STRATEGY = 'median'
OR_STRATEGY = 'sum'

COVERAGE = 0.8
MIN_METACELLS = 3
BUDGET_TOLERANCE = 0.9       # metacells below 90% of their UMI budget count as under budget
THIN_FRACTIONS = [0.5, 0.25] # keep 50% and 25% of UMIs
REFERENCE_SIZE = 50          # current SEACells target_metacell_size
RUN_SEACELLS = False         # set True only if SEACells is installed in the environment
SEED = 0

# --- sizing sweep -------------------------------------------------------------------------------
# The two knobs that change how genes are weighted when the UMI budget is set. Which is right is an
# empirical question, so run all four and compare rather than choosing up front.
#   split_isozymes      True  = each isozyme branch must be detectable in its own right (what
#                               calculate_ecs scores); False = weight an isozyme by its share of the
#                               summed reaction, which is what a flux bound actually depends on.
#   sizing_and_strategy 'min' = size for every subunit of a complex (the faithful AND);
#                               'median' = size for the typical subunit, robust to residual dropout.
SIZING_VARIANTS = {
    'split=T,and=median': dict(split_isozymes=True,  sizing_and_strategy='median'),
    'split=F,and=median': dict(split_isozymes=False, sizing_and_strategy='median'),
    'split=T,and=min':    dict(split_isozymes=True,  sizing_and_strategy='min'),
    'split=F,and=min':    dict(split_isozymes=False, sizing_and_strategy='min'),
}
REFERENCE_VARIANT = 'split=T,and=median'   # current behaviour; Parts 1 and 3 describe this one in detail

# Scoring is deliberately NOT swept. Every variant is measured with the same ruler, or the columns
# would not be comparable. Set this to how the scores will actually be read downstream:
# False scores the summed reaction (flux bounds), True scores each isozyme branch (ECS).
SCORING_SPLIT_ISOZYMES = False

sys.path.insert(0, REPO_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from metabolic_tools.metacell_diagnostics import recover_counts
from metabolic_tools.metabolic_metacells import metabolic_metacells
from metabolic_tools.metacell_benchmark import thin_counts, fixed_size_labels, seacells_labels, evaluate_grouping

adata = sc.read_h5ad(DATA_PATH)
adata.obs[SAMPLE_COL] = adata.obs_names.str.split('_E16').str[0]
report = recover_counts(adata)
print('Count recovery integer fraction:', report['integer_fraction'])
print(adata.obs[SAMPLE_COL].value_counts().to_string())

## Part 1 · Build metabolic metacells on the full data

In [ ]:
common = dict(celltype_col=CELLTYPE_COL, sample_col=SAMPLE_COL, species=SPECIES, symbol_col=SYMBOL_COL,
              and_strategy=AND_STRATEGY, or_strategy=OR_STRATEGY, coverage=COVERAGE,
              min_metacells=MIN_METACELLS, budget_tolerance=BUDGET_TOLERANCE, random_state=SEED)

# Build each sizing variant. The first run computes the PCA; passing its `cells` on to the next run
# means every variant groups on the SAME embedding, so the only thing differing is the sizing.
runs, cells = {}, adata
for name, kw in SIZING_VARIANTS.items():
    mc_v, info_v = metabolic_metacells(cells, **common, **kw)
    cells = info_v['cells']
    runs[name] = (mc_v, info_v)
    print(f'{name:<20} {cells.n_obs} cells -> {mc_v.n_obs} metacells; '
          f'{int(mc_v.obs["under_budget"].sum())} under budget')

mc, info = runs[REFERENCE_VARIANT]
print('\nReference variant:', REFERENCE_VARIANT)
print('Metacells mixing samples:', int((mc.obs['sample_purity'] < 1).sum()),
      '| mixing cell types:', int((mc.obs['celltype_purity'] < 1).sum()))

sizing_summary = pd.DataFrame({
    name: {'metacells': int(mc_v.n_obs),
           'median_cells': float(mc_v.obs['n_cells'].median()),
           'mean_budget_umis': float(mc_v.obs['umi_budget'].mean()),
           'under_budget': int(mc_v.obs['under_budget'].sum()),
           'detected_reachable': float(mc_v.obs['detected_leverage_reachable'].median()),
           **{f'rxn_{k}': int((info_v['reaction_classes']['class'] == k).sum())
              for k in ['pooled', 'partial', 'uncertain', 'off']}}
    for name, (mc_v, info_v) in runs.items()}).T
display(sizing_summary.round(3))
info['targets'][['n_cells', 'median_library_size', 'cap_cells', 'target_cells', 'target_umis']]

In [ ]:
# Metacells per cell type and sample (reference variant)
per_stratum = (mc.obs.groupby([CELLTYPE_COL, SAMPLE_COL], observed=True)
               .agg(metacells=('n_cells', 'size'), cells=('n_cells', 'sum'), median_cells=('n_cells', 'median'),
                    min_budget_ratio=('budget_ratio', 'min'), under_budget=('under_budget', 'sum'),
                    detected_reachable=('detected_leverage_reachable', 'median'))
               .round(2))
per_stratum

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.hist(np.log2(mc.obs['budget_ratio']), bins=40, color='grey')
ax.axvline(np.log2(BUDGET_TOLERANCE), color='tab:red', ls='--', lw=1)
ax.set_xlabel('log2(total UMIs / budget)')
ax.set_ylabel('metacells')
ax.set_title(f'UMI budget attainment, {REFERENCE_VARIANT} (left of red = under budget)', fontsize=10)

# 'reachable' = pooled + partial genes. Plotted rather than 'pooled' alone, which sits near 1.0 for
# every metacell by construction and so has almost nothing to compare against the prediction.
ax = axes[1]
colors = np.where(mc.obs['under_budget'], 'tab:red', 'tab:blue')
ax.scatter(mc.obs['expected_detected_leverage_reachable'], mc.obs['detected_leverage_reachable'],
           c=colors, s=18, alpha=0.7, linewidths=0)
ax.plot([0, 1], [0, 1], color='grey', ls='--', lw=0.8)
ax.set_xlabel('predicted detection (pooled + partial genes)')
ax.set_ylabel('observed detection (pooled + partial genes)')
ax.set_title('Does the dropout model predict detection? (red = under budget)', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Gene and reaction classes per cell type
gene_class_counts = pd.crosstab(info['gene_classes']['group'], info['gene_classes']['class'])
reaction_class_counts = pd.crosstab(info['reaction_classes']['group'], info['reaction_classes']['class'])
display(gene_class_counts)
display(reaction_class_counts)

## Part 2 · Benchmark on thinned data
For each thinning level, **every sizing variant is re-run on the thinned cells**, so budgets adapt to the lower depth. Each grouping is then compared with **the same groups at full depth**:
- `false_zero_rate`: reaction features present at full depth that read zero after thinning. This is the dropout damage we want to minimise.
- `median_rel_error`, `spearman`: how closely scores track full depth.
- `detected_leverage_all` / `_reachable`: leverage-weighted share of model genes detected, over all genes and over pooled + partial.
- `compactness`: how similar the cells in a metacell are (lower is tighter; 1 = as varied as the whole cell type).
- `sample_purity < 1`: the metacell mixes samples.

**Reading the sweep.** Bigger metacells always look better on the error metrics, so a variant is only genuinely better if it improves `false_zero_rate` *without* simply making fewer, larger metacells — read those columns together with `metacells` and `compactness`. The question each knob answers:
- `split=T` vs `split=F`: does insisting each isozyme branch be individually detectable buy accuracy, or just cost metacells? On synthetic data `split=F` gave ~8% smaller budgets and slightly more metacells.
- `and=min` vs `and=median`: sizing for every complex subunit raises the budget if subunits are harder to detect than average, and *lowers* it if they are easier — in which case single-gene reactions get squeezed. Which way it goes here is the thing to find out; mitochondrial subunits are often well expressed.

The `fixed_*` rows are references, not candidates: they show what the UMI budgeting and sample separation are buying over a plain cells-per-metacell rule.

In [ ]:
records = []
thin_runs = {}
for frac in THIN_FRACTIONS:
    print(f'--- keeping {frac:.0%} of UMIs ---')
    thinned = thin_counts(cells, frac, random_state=SEED)

    # Re-size every variant on the thinned cells, so budgets adapt to the lower depth. As in Part 1,
    # the embedding is computed once and shared, so the variants differ only in their sizing.
    var_runs = {}
    for name, kw in SIZING_VARIANTS.items():
        mc_t, info_t = metabolic_metacells(thinned, **common, **kw)
        thinned = info_t['cells']
        var_runs[name] = (mc_t, info_t)
        print(f'   sized {name:<20} -> {mc_t.n_obs} metacells')
    thin_runs[frac] = var_runs

    # Each variant, plus the two fixed-size references matched to the reference variant.
    groupings = [(name, info_t['labels'], info_t) for name, (mc_t, info_t) in var_runs.items()]
    ref_mc, ref_info = var_runs[REFERENCE_VARIANT]
    matched_size = (thinned.obs[CELLTYPE_COL].astype(str).value_counts()
                    / ref_mc.obs[CELLTYPE_COL].value_counts()).to_dict()
    groupings.append(('fixed_size_matched',
                      fixed_size_labels(thinned, CELLTYPE_COL, matched_size, random_state=SEED), ref_info))
    groupings.append((f'fixed_{REFERENCE_SIZE}',
                      fixed_size_labels(thinned, CELLTYPE_COL, REFERENCE_SIZE, random_state=SEED), ref_info))
    if RUN_SEACELLS:
        groupings.append((f'seacells_{REFERENCE_SIZE}',
                          seacells_labels(thinned, CELLTYPE_COL, REFERENCE_SIZE), ref_info))

    for name, labels, info_t in groupings:
        print(f'   scoring {name}')
        records.append(evaluate_grouping(cells, thinned, labels, name, frac, CELLTYPE_COL, SAMPLE_COL,
                                         info_t['targets']['target_umis'], info_t['genes'],
                                         info_t['gene_classes'], species=SPECIES,
                                         and_strategy=AND_STRATEGY, or_strategy=OR_STRATEGY,
                                         split_isozymes=SCORING_SPLIT_ISOZYMES,
                                         budget_tolerance=BUDGET_TOLERANCE))
bench = pd.concat(records, ignore_index=True)
bench['is_variant'] = bench['method'].isin(SIZING_VARIANTS)
print('done:', bench.shape)

In [ ]:
# The four variants are the sweep; the fixed_* rows are references, not candidates.
summary = (bench.groupby(['fraction', 'method'])
           .agg(metacells=('metacell', 'size'),
                median_cells=('n_cells', 'median'),
                false_zero_rate=('false_zero_rate', 'median'),
                median_rel_error=('median_rel_error', 'median'),
                spearman=('spearman', 'median'),
                detected_leverage_all=('detected_leverage_all', 'median'),
                detected_leverage_reachable=('detected_leverage_reachable', 'median'),
                compactness=('compactness', 'median'),
                mixed_sample_share=('sample_purity', lambda s: float((s < 1).mean())),
                under_budget_share=('under_budget', 'mean'))
           .round(3))
# order the sweep first, references after
order = list(SIZING_VARIANTS) + [m for m in summary.index.get_level_values('method').unique()
                                 if m not in SIZING_VARIANTS]
summary = summary.reindex(pd.MultiIndex.from_product([THIN_FRACTIONS, order],
                                                     names=['fraction', 'method'])).dropna(how='all')
summary

In [ ]:
# False-zero rate by cell type (median over metacells), at each thinning level
by_type = bench.pivot_table(index=['fraction', CELLTYPE_COL], columns='method', values='false_zero_rate', aggfunc='median').round(3)
by_type

## Part 3 · Under-budget metacells
Under-budget metacells come from cell type × sample combinations too small to reach the UMI budget. Three questions:
1. **How much sparser are they** than at-budget metacells of the same cell type?
2. **Does sparsity rise smoothly as budget attainment falls**, and at what point does it get bad? That tells us whether 0.9 is the right flag threshold.
3. **Does the dropout model predict their detection?** If so, the predicted detection can flag unreliable metacells without needing a benchmark.

In [ ]:
# 1. Under-budget vs at-budget metacells, within the same cell type (reference variant)
m = bench[bench['method'] == REFERENCE_VARIANT]
paired = (m.groupby(['fraction', CELLTYPE_COL, 'under_budget'])
           .agg(metacells=('metacell', 'size'), false_zero_rate=('false_zero_rate', 'median'),
                detected_leverage_all=('detected_leverage_all', 'median'), budget_ratio=('budget_ratio', 'median'))
           .unstack('under_budget'))
paired = paired.reindex(columns=pd.MultiIndex.from_product(
    [['metacells', 'false_zero_rate', 'detected_leverage_all', 'budget_ratio'], [False, True]]))
has_both = paired['metacells'].notna().all(axis=1)
comparison = pd.DataFrame({
    'at_budget_metacells': paired[('metacells', False)],
    'under_budget_metacells': paired[('metacells', True)],
    'under_budget_ratio': paired[('budget_ratio', True)],
    'false_zero_at': paired[('false_zero_rate', False)],
    'false_zero_under': paired[('false_zero_rate', True)],
    'detected_at': paired[('detected_leverage_all', False)],
    'detected_under': paired[('detected_leverage_all', True)],
})
comparison['false_zero_increase'] = comparison['false_zero_under'] - comparison['false_zero_at']
comparison['detection_drop'] = comparison['detected_at'] - comparison['detected_under']
print('Cell types with both under- and at-budget metacells:')
comparison[has_both].round(3)

In [ ]:
# The under-budget metacells themselves
cols = ['fraction', 'metacell', CELLTYPE_COL, SAMPLE_COL, 'n_cells', 'total_umis', 'budget_ratio',
        'false_zero_rate', 'detected_leverage_all', 'detected_leverage_reachable',
        'expected_detected_leverage_reachable']
m[m['under_budget']][cols].sort_values(['fraction', 'budget_ratio']).round(3)

In [ ]:
# 2. Sparsity against budget attainment (each point = one metacell)
#    Sweep variants are drawn as circles, the fixed-size references as other markers.
ref_markers = {'fixed_size_matched': 's', f'fixed_{REFERENCE_SIZE}': '^', f'seacells_{REFERENCE_SIZE}': 'D'}
fig, axes = plt.subplots(2, len(THIN_FRACTIONS), figsize=(6 * len(THIN_FRACTIONS), 8), squeeze=False)
for c, frac in enumerate(THIN_FRACTIONS):
    for r, metric in enumerate(['false_zero_rate', 'detected_leverage_all']):
        ax = axes[r, c]
        for method, d in bench[bench['fraction'] == frac].groupby('method'):
            ax.scatter(d['budget_ratio'], d[metric], s=14, alpha=0.5,
                       marker=ref_markers.get(method, 'o'), label=method, linewidths=0)
        ax.axvline(BUDGET_TOLERANCE, color='tab:red', ls='--', lw=0.8)
        ax.axvline(1, color='grey', ls=':', lw=0.8)
        ax.set_xscale('log')
        ax.set_xlabel('total UMIs / budget')
        ax.set_ylabel(metric)
        ax.set_title(f'{frac:.0%} of UMIs kept', fontsize=10)
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.show()

bins = [0, 0.25, 0.5, 0.75, 0.9, 1.1, 1.5, 2, np.inf]
bench['budget_bin'] = pd.cut(bench['budget_ratio'], bins)
bench.pivot_table(index=['fraction', 'budget_bin'], columns='method', values='false_zero_rate', aggfunc='median', observed=True).round(3)

In [ ]:
# 3. Is predicted detection a reliable warning? (reference variant, pooled + partial genes)
fig, axes = plt.subplots(1, len(THIN_FRACTIONS), figsize=(5 * len(THIN_FRACTIONS), 4.2), squeeze=False)
for ax, frac in zip(axes.flat, THIN_FRACTIONS):
    d = m[m['fraction'] == frac]
    ax.scatter(d['expected_detected_leverage_reachable'], d['detected_leverage_reachable'],
               c=np.where(d['under_budget'], 'tab:red', 'tab:blue'), s=18, alpha=0.7, linewidths=0)
    ax.plot([0, 1], [0, 1], color='grey', ls='--', lw=0.8)
    err = (d['detected_leverage_reachable'] - d['expected_detected_leverage_reachable'])
    ax.set_title(f'{frac:.0%} kept · mean observed - predicted = {err.mean():+.3f}', fontsize=9)
    ax.set_xlabel('predicted detection')
    ax.set_ylabel('observed detection')
plt.tight_layout()
plt.show()

for frac in THIN_FRACTIONS:
    d = m[m['fraction'] == frac]
    for flag, g in d.groupby('under_budget'):
        e = g['detected_leverage_reachable'] - g['expected_detected_leverage_reachable']
        print(f'{frac:.0%} kept | under_budget={flag}: n={len(g)}, mean error {e.mean():+.3f}, '
              f'corr(predicted, false_zero_rate) = {g["expected_detected_leverage_reachable"].corr(g["false_zero_rate"]):+.2f}')

### How to read Part 3
- **Small `false_zero_increase`** (a few points) means under-budget metacells are usable, just flagged. **Large** means consider dropping them, or pooling that cell type across samples instead.
- In the budget plot, the ratio where the false-zero rate **starts climbing steeply** is a better under-budget threshold than 0.9, if it differs.
- If **predicted and observed detection agree** (points near the diagonal, including red ones), the predicted detection stored in each metacell's `obs` can serve as a per-metacell reliability score downstream.

## Save

In [ ]:
# Reference variant in full, plus the sizing comparison and per-variant classes
mc.write_h5ad(os.path.join(OUT_DIR, 'metabolic_metacells_E16.h5ad'))
info['labels'].rename('metacell').to_csv(os.path.join(OUT_DIR, 'cell_to_metacell.csv'))
info['targets'].to_csv(os.path.join(OUT_DIR, 'size_targets.csv'))
info['gene_classes'].to_csv(os.path.join(OUT_DIR, 'gene_classes.csv'), index=False)
info['reaction_classes'].to_csv(os.path.join(OUT_DIR, 'reaction_classes.csv'), index=False)
per_stratum.to_csv(os.path.join(OUT_DIR, 'metacells_per_stratum.csv'))

sizing_summary.to_csv(os.path.join(OUT_DIR, 'sizing_sweep_summary.csv'))
for name, (mc_v, info_v) in runs.items():
    tag = name.replace('=', '').replace(',', '_')
    info_v['targets'].to_csv(os.path.join(OUT_DIR, f'size_targets_{tag}.csv'))
    info_v['reaction_classes'].to_csv(os.path.join(OUT_DIR, f'reaction_classes_{tag}.csv'), index=False)

bench.drop(columns=['budget_bin']).to_csv(os.path.join(OUT_DIR, 'benchmark_per_metacell.csv'), index=False)
summary.to_csv(os.path.join(OUT_DIR, 'benchmark_summary.csv'))
by_type.to_csv(os.path.join(OUT_DIR, 'benchmark_false_zero_by_celltype.csv'))
comparison.to_csv(os.path.join(OUT_DIR, 'under_budget_comparison.csv'))
with open(os.path.join(OUT_DIR, 'settings.json'), 'w') as f:
    json.dump({'sizing_variants': {k: v for k, v in SIZING_VARIANTS.items()},
               'reference_variant': REFERENCE_VARIANT,
               'scoring_split_isozymes': SCORING_SPLIT_ISOZYMES,
               'and_strategy': AND_STRATEGY, 'or_strategy': OR_STRATEGY,
               'coverage': COVERAGE, 'min_metacells': MIN_METACELLS,
               'budget_tolerance': BUDGET_TOLERANCE, 'thin_fractions': THIN_FRACTIONS,
               'seed': SEED}, f, indent=2)
print('Saved to', OUT_DIR)

## Sending results back
1. **File → Save Notebook As…** → `metabolic/results/03_metabolic_metacells_E16/03_metabolic_metacells_E16_run.ipynb`
2. In a terminal (the `.h5ad` is left out of the zip to keep it small):
```bash
cd /scratch/prj/crb_inner_ear/k2147692/metabolic/results && rm -f 03_metabolic_metacells_E16.zip && zip -r 03_metabolic_metacells_E16.zip 03_metabolic_metacells_E16 -x '*.h5ad'
cd /scratch/prj/crb_inner_ear/k2147692/metabolic/code/Metabolic-pipeline && git checkout -- notebooks/
```
3. Right-click the zip in Jupyter → **Download**, then extract into `Documents\Metabolic-results`.